In [ ]:
# --- PocketLM setup (works in Colab and locally) ---
try:
    import pocketlm  # already installed locally / in CI
except ImportError:
    # Colab: install straight from GitHub. The corpus ships *inside* the package,
    # so there is no repo to clone and no working directory to change.
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "git+https://github.com/ni5h4nt/pocketlm"])
    import pocketlm  # noqa: F811

import os
import torch

DEVICE = "micro" if os.environ.get("POCKETLM_CI") else ("cuda" if torch.cuda.is_available() else "cpu")
CORPUS_PATH = pocketlm.corpus_path()   # packaged data: valid from any directory
print("device:", DEVICE, "(timing is guidance, not a contract)")

# l3.3 Causal masking

A decoder predicts the next token, so a token must never peek at the future.
The causal mask sets future scores to -inf before the softmax, zeroing their
weight.

In [ ]:
from pocketlm import scaled_dot_product_attention

torch.manual_seed(0)
q = torch.randn(1, 4, 8)
k = torch.randn(1, 4, 8)
v = torch.randn(1, 4, 8)
out, w = scaled_dot_product_attention(q, k, v, causal=True)
print("causal weights (upper triangle is 0):")
print(w[0].round(decimals=2))

Proof, not promise: edit every token after the first and the first token's
output does not move. The past cannot see the future.

In [ ]:
k2 = k.clone()
v2 = v.clone()
k2[:, 1:] += 5.0
v2[:, 1:] += 5.0
out2, _ = scaled_dot_product_attention(q, k2, v2, causal=True)
print("position 0 unchanged by future edits:", torch.allclose(out[:, 0], out2[:, 0]))
upper = torch.triu(torch.ones(4, 4), diagonal=1).bool()
assert torch.all(w[0][upper] == 0)
assert torch.allclose(out[:, 0], out2[:, 0], atol=1e-6)